# Graph RAG - 检索与生成

> 《奥本海默》中饰演奥本海默的演员，在资料里的另一部电影中饰演了谁？

## 准备阶段

In [12]:
from pathlib import Path

import graphrag.api as api
import pandas as pd
from graphrag.config.models.graph_rag_config import GraphRagConfig
from graphrag_llm.config import ModelConfig
from IPython.display import Markdown, display

project_dir = Path.cwd()
if project_dir.name == "课件":
    project_dir = project_dir.parent
work_dir = project_dir / "resource" / "graph_rag"
output = work_dir / "output"

text_units = pd.read_parquet(output / "text_units.parquet")
entities = pd.read_parquet(output / "entities.parquet")
relationships = pd.read_parquet(output / "relationships.parquet")
communities = pd.read_parquet(output / "communities.parquet")
reports = pd.read_parquet(output / "community_reports.parquet")

print(f"原文切片 {len(text_units)} 段")

原文切片 3 段


In [13]:
config = GraphRagConfig()
config.vector_store.db_uri = str(output / "lancedb")
config.reporting.base_dir = str(work_dir / "logs")
dimensions = 1024
config.vector_store.vector_size = dimensions
for schema in config.vector_store.index_schema.values():
    schema.vector_size = dimensions

config.completion_models["default_completion_model"] = ModelConfig(
    model_provider="openai", 
    model="chat-standard",
    call_args={"extra_body": {"enable_thinking": False}},
    api_base="http://localhost:4000/v1", 
    api_key="sk-zCjzfRyCk5KGoEzLvhK20Q",
)
config.embedding_models["default_embedding_model"] = ModelConfig(
    model_provider="openai", 
    model="text-embedding",
    api_base="http://localhost:4000/v1",
    api_key="sk-zCjzfRyCk5KGoEzLvhK20Q",
)

## 1. Basic Search · 直接找原文

先用熟悉的向量检索作对照。Basic 把**问题和原文切片**按向量相似度匹配，把选中的切片交给模型写答案。题目只提到《奥本海默》，要答出另一部影片里的角色，模型需要同时看到哪两段资料？

这次取两段切片。运行后先读回答，再看它拿到了哪两段原文。

In [14]:
casting_question = "《奥本海默》中饰演奥本海默的演员，在资料里的另一部电影中饰演了谁？请写出两部片名、演员和对应角色。"
answer_style = "用中文简短回答，保留证据编号。材料不足时指出缺少哪一段证据。"
config.basic_search.k = 2

basic_answer, basic_context = await api.basic_search(
    config=config,
    text_units=text_units,
    query=casting_question,
    response_type=answer_style,
)
display(Markdown(basic_answer))

## 回答

在《奥本海默》中饰演物理学家罗伯特·奥本海默的演员是**希里安·墨菲** [Data: Sources (2)]。

在资料中的另一部电影《盗梦空间》里，他饰演的角色是**罗伯特·费舍尔** [Data: Sources (0)]。

### 两部影片信息汇总

| 片名 | 演员 | 对应角色 |
|------|------|----------|
| 《奥本海默》 | 希里安·墨菲 | 罗伯特·奥本海默 |
| 《盗梦空间》 | 希里安·墨菲 | 罗伯特·费舍尔 |

以上信息均可从现有材料中获得，无需补充其他证据。

In [ ]:
print(basic_context["sources"].to_string(index=False))

核对答案时，一段原文应当能确认希里安·墨菲饰演了奥本海默，另一段应当能确认他在《盗梦空间》里饰演费舍尔。如果两段都找到了，Basic 已经能答对；如果少了一段，即使回答里写出了正确片名，也要区分哪一半有依据。

## 2. Local Search · 围绕实体找证据

保持问题不变，把向量匹配的对象换成**实体描述**。例如，《奥本海默》这部电影和罗伯特·奥本海默这个角色都可能成为入口。Local 用匹配到的实体收集相关关系、原文切片和社区报告，在上下文预算内选取材料，再让模型作答。这里最值得找的是能把两份原文接起来的演员。

In [15]:
local_answer, local_context = await api.local_search(
    config=config,
    entities=entities,
    relationships=relationships,
    text_units=text_units,
    communities=communities,
    community_reports=reports,
    community_level=0,
    covariates=None,  # 第 44 节没有单独抽取声明
    query=casting_question,
    response_type=answer_style,
)
display(Markdown(local_answer))

## 回答

在《奥本海默》中饰演罗伯特·奥本海默的演员是**希里安·墨菲**。根据资料，他在另一部电影《盗梦空间》中饰演了**罗伯特·费舍尔**（也称费舍尔）[Data: Entities (5, 8); Relationships (7, 43); Sources (0, 2)]。

### 两部电影、演员及对应角色

| 电影 | 演员 | 饰演角色 |
|------|------|----------|
| 《奥本海默》 | 希里安·墨菲 | 罗伯特·奥本海默 [Data: Entities (5, 21); Relationships (43); Sources (2)] |
| 《盗梦空间》 | 希里安·墨菲 | 罗伯特·费舍尔 [Data: Entities (5, 8); Relationships (7); Sources (0)] |

先找这次上下文里与希里安·墨菲直接相连的关系，再读它附带的原文。关系的两端可能是演员和影片，也可能是演员和角色，具体以这次索引为准。

In [16]:
edges = local_context["relationships"]
actor = "希里安·墨菲"
actor_edges = edges["source"].str.contains(actor) | edges["target"].str.contains(actor)
display(edges.loc[actor_edges, ["id", "source", "target", "description"]])

for row in local_context["sources"].itertuples():
    print(f"[{row.id}] {row.text}\n")

,id,source,target,description
1,42,希里安·墨菲,《奥本海默》,希里安·墨菲参演《奥本海默》。
3,54,《奥本海默》,希里安·墨菲,《奥本海默》由希里安·墨菲参演。
12,7,希里安·墨菲,罗伯特·费舍尔,在《盗梦空间》中饰演罗伯特·费舍尔，原文也称费舍尔。
14,43,希里安·墨菲,罗伯特·奥本海默,希里安·墨菲在《奥本海默》中饰演罗伯特·奥本海默。
15,57,罗伯特·奥本海默,希里安·墨菲,罗伯特·奥本海默由希里安·墨菲饰演。
26,6,希里安·墨菲,《盗梦空间》,希里安·墨菲参演《盗梦空间》，在片中饰演罗伯特·费舍尔。


[2] 《奥本海默》于2023年上映，克里斯托弗·诺兰担任导演和编剧，路德维希·戈兰松负责配乐。

希里安·墨菲饰演物理学家罗伯特·奥本海默，艾米莉·布朗特饰演他的妻子凯蒂。马特·达蒙饰演负责曼哈顿计划的莱斯利·格罗夫斯将军，他任命奥本海默领导洛斯阿拉莫斯实验室。


[0] 《盗梦空间》于2010年上映，克里斯托弗·诺兰担任导演和编剧，汉斯·季默负责配乐。

莱昂纳多·迪卡普里奥饰演能够潜入他人梦境的柯布，约瑟夫·高登-莱维特饰演他的搭档亚瑟。两人接下的任务，是把一个想法植入罗伯特·费舍尔的梦中。希里安·墨菲饰演费舍尔。


[1] 《星际穿越》于2014年上映，由克里斯托弗·诺兰执导，克里斯托弗·诺兰与乔纳森·诺兰共同编剧，汉斯·季默负责配乐。

马修·麦康纳饰演飞行员库珀，安妮·海瑟薇饰演科学家阿米莉亚·布兰德。两人穿越虫洞，为人类寻找新的家园。杰西卡·查斯坦饰演成年后的墨菲，她是库珀留在地球上的女儿。




把两个答案并排想一想。如果都答对了，Local 多给出了哪些可以核对的关系？如果上表为空，先在切片里找演员姓名，再看实体表里用了什么写法。关系未进入这次上下文，仍然可以回到原文核实。

## 3. Global Search · 从社区报告归纳片库

> 三部电影在导演、编剧和配乐上有哪些稳定的合作，又有哪些变化？

这次没有指定从哪位创作者查起。Global 可以从上一节生成的**社区报告**入手。这里固定读取顶层报告，按上下文容量分批。**Map** 阶段让模型从每批报告中提取与问题有关的要点并评分；**Reduce** 阶段汇集得分较高、能放进上下文的要点，组织成最终回答。报告少时，一批就可能装下。

In [17]:
global_answer, global_context = await api.global_search(
    config=config,
    entities=entities,
    communities=communities,
    community_reports=reports,
    community_level=0,
    dynamic_community_selection=False,
    query="比较资料中三部电影的导演、编剧与配乐分工。哪些人反复合作，哪里换了人？用片名举例。",
    response_type="用中文按导演、编剧、配乐分三点回答，注明报告编号。资料未说明的地方请指出。",
)
display(Markdown(global_answer))

# 三部电影导演、编剧与配乐分工比较

根据提供的资料，以下按导演、编剧、配乐三个方面对《星际穿越》《盗梦空间》和《奥本海默》三部电影的幕后创作团队进行比较分析。

---

## 一、导演

- **《星际穿越》**：由克里斯托弗·诺兰担任导演 [Data: Reports (2, 0)]。
- **《盗梦空间》**：同样由克里斯托弗·诺兰独自负责导演工作 [Data: Reports (2, 0)]。
- **《奥本海默》**：**资料未说明**。提供的资料中未被明确提及该片的导演信息，因此无法确认克里斯托弗·诺兰是否参与了该片的导演工作，也无法判断该片在导演层面是否与前两部作品存在人员延续或更换 [Data: Reports (1)]。

**反复合作情况**：在已知信息范围内，《星际穿越》与《盗梦空间》在导演岗位上保持了高度一致的合作班底，均由克里斯托弗·诺兰执导 [Data: Reports (2, 0)]。

---

## 二、编剧

- **《星际穿越》**：由克里斯托弗·诺兰与乔纳森·诺兰共同担任编剧 [Data: Reports (2, 0)]。
- **《盗梦空间》**：由克里斯托弗·诺兰独自负责编剧工作 [Data: Reports (2, 0)]。
- **《奥本海默》**：**资料未说明**。提供的资料中未被明确提及该片的编剧信息，因此无法确认克里斯托弗·诺兰或其他人是否参与了该片的编剧工作，也无法判断该片在编导层面是否与前两部作品存在人员延续或更换 [Data: Reports (1)]。

**反复合作情况**：克里斯托弗·诺兰是资料中唯一反复合作的核心幕后创作者，他在《星际穿越》与《盗梦空间》中均担任了编剧。其中，《星际穿越》引入了乔纳森·诺兰作为联合编剧，而《盗梦空间》则由克里斯托弗·诺兰独立完成编剧 [Data: Reports (2, 0)]。

---

## 三、配乐

- **《星际穿越》**：由汉斯·季默负责配乐 [Data: Reports (2, 1)]。
- **《盗梦空间》**：同样由汉斯·季默负责配乐 [Data: Reports (2, 1)]。
- **《奥本海默》**：配乐工作换由路德维希·戈兰松负责 [Data: Reports (2, 1)]。

**反复合作与换人情况**：
- **反复合作**：汉斯·季默负责了《星际穿越》和《盗梦空间》的配乐，成为继克里斯托弗·诺兰之后另一位将这两部电影联系在一起的关键人物 [Data: Reports (2, 1)]。
- **换人**：配乐方面存在明显的换人情况，《奥本海默》的配乐师由汉斯·季默更换为路德维希·戈兰松 [Data: Reports (2, 1)]。

---

## 总结

从跨作品的创作网络来看，《星际穿越》与《盗梦空间》在导演（克里斯托弗·诺兰）和配乐（汉斯·季默）两个关键岗位上保持了高度一致的合作班底；克里斯托弗·诺兰同时兼任这两部影片的编剧，是资料中唯一反复合作的核心幕后创作者 [Data: Reports (2, 0, 1)]。

对于《奥本海默》，在已知信息中仅确认了配乐师的更换（由路德维希·戈兰松接替汉斯·季默），而其导演与编剧团队的构成**资料均未说明**，无法做出进一步判断 [Data: Reports (2, 0, 1)]。

查看本轮的报告标题，再选“配乐”这一点对照原文。原文中《盗梦空间》和《星际穿越》都写了汉斯·季默，《奥本海默》写的是路德维希·戈兰松。如果回答少举了一部片，读对应的报告，看看这个细节在生成报告时是否就已经被省略了。

In [18]:
used_reports = global_context["reports"]
display(used_reports[["id", "title"]])
print(used_reports.iloc[0]["content"])  # 改行号可读另一份报告

,id,title
0,2,《星际穿越》电影社区报告
1,0,《盗梦空间》电影社区报告
2,1,《奥本海默》电影社区报告


# 《星际穿越》电影社区报告

本报告以2014年上映的科幻电影《星际穿越》为核心，梳理了该片在导演、编剧、配乐、演员及角色层面的创作网络。克里斯托弗·诺兰作为核心创作者，通过执导与编剧将《星际穿越》与其另一部作品《盗梦空间》紧密相连；马修·麦康纳、安妮·海瑟薇等演员则通过饰演库珀、阿米莉亚·布兰德等关键角色，共同构建了影片的故事脉络。

## 诺兰兄弟的编导核心与跨作品联系

克里斯托弗·诺兰是连接本社区多部作品的核心人物。他担任了电影《星际穿越》的导演，并与乔纳森·诺兰共同担任该片的编剧 [Data: Entities (1, 9, 10); Relationships (29, 15)]。此外，克里斯托弗·诺兰还在电影《盗梦空间》中担任导演和编剧，从而在《星际穿越》与《盗梦空间》之间建立了明确的创作联系 [Data: Entities (1); Relationships (0)]。

## 汉斯·季默的配乐纽带作用

作曲家汉斯·季默负责了电影《星际穿越》的配乐工作 [Data: Entities (2, 9)]。同时，资料表明他也负责了电影《盗梦空间》的配乐 [Data: Entities (2); Relationships (1)]。这使得汉斯·季默成为继克里斯托弗·诺兰之后，另一位将《星际穿越》与《盗梦空间》这两部电影联系在一起的关键幕后创作者。

## 主要演员与角色的对应关系

电影《星际穿越》的叙事由几位核心演员及其饰演的角色支撑。马修·麦康纳在片中饰演飞行员库珀，该角色穿越虫洞为人类寻找新的家园 [Data: Entities (11, 14); Relationships (18, 24)]。安妮·海瑟薇饰演科学家阿米莉亚·布兰德，她与库珀一同穿越虫洞寻找新家园 [Data: Entities (12, 15); Relationships (20, 25)]。杰西卡·查斯坦则饰演成年后的墨菲，她是库珀留在地球上的女儿 [Data: Entities (13, 16); Relationships (22, 26)]。这三位演员均参演了《星际穿越》，通过各自的角色分工构成了影片的父女情感线与太空探索主线 [Data: Relationships (18, 20, 22)]。

## 《星际穿越》作为社区绝对枢纽

整个社区的网络结构高度围绕